# 04 — Index B (Steady-State)

**Objetivo.** Confirmar que el baseline plano ya resuelve Index_B sin necesidad de NN.

**Pistas del enunciado.** Steady-State defensivo — baja volatilidad, comportamiento estable.

**Nivel de esfuerzo.** BAJO. Hipótesis: `baseline_flat` lo clava. Dedicar tiempo mínimo y confirmar.

**Estrategia.** Verificar visualmente el rollout del baseline flat. Si el RMSE es razonable, listo. NN solo si el baseline falla sorpresivamente.

**Qué esperamos.** RMSE muy bajo con flat. Si hay ligero drift, probar `baseline_drift`.

**Qué NO hace.** No explora arquitecturas complejas. No entrena NN salvo que el backtest obligue.

**Inputs.** `data/train_indices.csv`, `results/baselines.json`

**Output esperado.** `results/index_B.json` (approach_type = `baseline_flat` o `baseline_drift`, sin model_path).

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

OWNER = "oscar"   # <-- cada miembro pone su nombre aquí
IDX   = "Index_B"

import json
import numpy as np
import matplotlib.pyplot as plt

from utils import (
    load_hackathon_data, make_temporal_split,
    baseline_flat, baseline_drift, baseline_random_walk,
    backtest_autoregressive, plot_rollout,
    DATA_DIR, VAL_DAYS, V_IN_SHARED
)

os.makedirs('results', exist_ok=True)

## 1. Carga + diagnóstico de Index_B

In [ ]:
data  = load_hackathon_data(DATA_DIR)
idx   = data['train_indices']
serie = idx[IDX].dropna().values

print(f'{IDX}: {len(serie)} días  |  min={serie.min():.0f}  max={serie.max():.0f}  último={serie[-1]:.0f}')

plt.figure(figsize=(13, 3))
plt.plot(serie, lw=0.8)
plt.title(f'{IDX} — precios brutos (esperamos baja volatilidad)')
plt.tight_layout()
plt.show()

## 2. Referencia: RMSE de baselines para B

In [ ]:
with open('results/baselines.json') as f:
    baselines = json.load(f)

print(f'Baselines para {IDX}:')
for bl, rmse in baselines[IDX].items():
    print(f'  {bl:<15} RMSE = {rmse:,.0f}')

best_bl   = min(baselines[IDX], key=baselines[IDX].get)
best_rmse = baselines[IDX][best_bl]
print(f'\nMejor baseline: {best_bl}  RMSE={best_rmse:,.0f}')

## 3. Confirmación visual del rollout plano

In [ ]:
# Correr el mejor baseline manualmente para visualizar el rollout
serie_train, _ = make_temporal_split(serie, val_days=VAL_DAYS, v_in=V_IN_SHARED)

if best_bl == 'flat':
    preds_best = baseline_flat(serie_train, VAL_DAYS)
elif best_bl == 'drift':
    preds_best = baseline_drift(serie_train, VAL_DAYS)
else:
    preds_best = baseline_random_walk(serie_train, VAL_DAYS)

plot_rollout(serie, preds_best, index_name=f'{IDX} — {best_bl}', val_days=VAL_DAYS)

## 4. Decisión — ¿el baseline basta?

Si el RMSE del baseline es aceptable (muy inferior al de A/F) y el rollout visual es razonable, B queda resuelto. Escribir el JSON y pasar al siguiente índice.

In [ ]:
# Mapa nombre → approach_type del JSON acordado
_tipo_map = {'flat': 'baseline_flat', 'drift': 'baseline_drift', 'random_walk': 'baseline_rw'}

info = {
    'index':              IDX,
    'owner':              OWNER,
    'approach_type':      _tipo_map[best_bl],
    'strategy':           best_bl,
    'rmse_backtest_252d': best_rmse,
    'model_path':         None,
    'log_ret_mode':       False,
    'v_in':               None,
    'n_features':         1,
    'aux_source':         None,
    'aux_test_source':    None,
    'aux_columns':        None,
    'ghost_source_index': None,
    'ghost_lag':          None,
    'notes':              'Baseline plano — serie defensiva, NN no aporta'
}

with open('results/index_B.json', 'w') as f:
    json.dump(info, f, indent=2)

print('Guardado: results/index_B.json')
print(json.dumps(info, indent=2))

## 5. [OPCIONAL] NN mínima — solo si el baseline falla sorpresivamente

Si el RMSE del baseline es inaceptablemente alto, intentar un LSTM mínimo. No usar esta sección por defecto.

In [ ]:
# [EJECUTAR SOLO SI EL BASELINE NO BASTA]
# from utils import make_window_dataset, build_model, train_model
# import tensorflow as tf
# ...
# Si se entrena un modelo: actualizar results/index_B.json con approach_type='nn'